In [ ]:
# ==========================================================
# Notebook 01 — Data Ingestion and Master Cache Construction
# ==========================================================
#
# Purpose:
# Load, standardise, categorise, and combine the extracted
# Department for Education source files into a single cached
# dictionary of core datasets for downstream processing.
#
# Inputs:
# - CSV files stored in:
#   - data/processed/
# - Supporting raw files if not yet copied into processed:
#   - workforce_turnover.csv
#   - workforce_ratios.csv
#
# Outputs:
# - master_data_cache.pkl
#   A dictionary of pandas DataFrames grouped by source type:
#   - census
#   - absence
#   - school_info
#   - spine
#   - swf
#   - ks4
#   - links
#
# Role in the pipeline:
# This notebook is the ingestion and harmonisation stage of
# the pipeline. It standardises year formats, normalises key
# identifiers, derives selected workforce metrics, groups
# related files into categories, and writes the combined
# datasets to a cached object used by later notebooks.
#
# Reproducibility note:
# The file paths in this notebook are currently configured for
# the author’s local machine. These should be updated if the
# project is run in a different environment.
#
# Important:
# This notebook was developed on a local Windows environment.
# Users reproducing the pipeline should replace absolute paths
# with environment-specific paths or relative project paths.
# ==========================================================

import pandas as pd
import os
import glob
import re
import pickle
import shutil
from pathlib import Path

# ==========================================================
# Configuration
# ==========================================================
#
# This section defines the working directories used for data
# ingestion and cache creation.
#
# - BASE_DIR points to the project data directory.
# - RAW_FOLDER contains original source files.
# - PROCESSED_FOLDER contains extracted / standardised CSVs.
# - CACHE_PATH is the output path for the combined cache.
#
# If reproducing this pipeline on another machine, update
# these paths before running the notebook.
# ==========================================================

# --- CONFIGURATION ---
BASE_DIR = Path(r'C:\Users\kiero\Documents\msc-dissertation\data')
RAW_FOLDER = BASE_DIR / 'raw'
PROCESSED_FOLDER = BASE_DIR / 'processed'
CACHE_PATH = PROCESSED_FOLDER / 'master_data_cache.pkl'

print(" Starting SMART Data Ingestion...\n")

# ==========================================================
# Helper Functions
# ==========================================================
#
# These helper functions standardise year formats, safely read
# CSV files with mixed encodings, normalise column names, and
# ensure key fields are renamed consistently across source files.
#
# This is necessary because Department for Education source
# files vary across years in both naming conventions and
# schema structure.
# ==========================================================

# --- HELPERS ---
def standardise_academic_year(val) -> str:
    """
    Convert many possible year formats to: 'YYYY-YYYY'
    Examples:
      '201617' -> '2016-2017'
      '2016/17' -> '2016-2017'
      '2016-17' -> '2016-2017'
      '2016-2017' -> '2016-2017'
      '2016/2017' -> '2016-2017'
    """
    if pd.isna(val):
        return None
    s = str(val).strip()

    # 6 digits: 201617
    if re.fullmatch(r"\d{6}", s):
        y1 = int(s[:4])
        y2 = int(s[:4]) + 1
        return f"{y1}-{y2}"

    # 2016/17 or 2016-17
    m = re.fullmatch(r"(20\d{2})[/-](\d{2})", s)
    if m:
        y1 = int(m.group(1))
        y2 = y1 + 1
        return f"{y1}-{y2}"

    # 2016/2017 or 2016-2017
    m = re.fullmatch(r"(20\d{2})[/-](20\d{2})", s)
    if m:
        y1 = int(m.group(1))
        y2 = int(m.group(2))
        return f"{y1}-{y2}"

    # If already looks like YYYY-YYYY, keep it
    if re.fullmatch(r"20\d{2}-20\d{2}", s):
        return s

    return s  # fallback (won't break pipeline)

def extract_year_from_filename(filename: str) -> str | None:
    """
    Extract _1617 / _1920 / _2324 pattern and convert to 'YYYY-YYYY'
    """
    m = re.search(r"_(\d{4})", filename)
    if not m:
        return None
    yc = m.group(1)  # e.g. 1617
    y1 = int("20" + yc[:2])
    y2 = int("20" + yc[2:])
    # Handles 1999 style? (not needed here) but safe.
    return f"{y1}-{y2}"

def safe_read_csv(path: Path) -> pd.DataFrame:
    try:
        return pd.read_csv(path, encoding="utf-8-sig", low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="cp1252", low_memory=False)

def upper_cols(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = [str(c).strip().upper() for c in df.columns]
    return df

def ensure_col(df: pd.DataFrame, wanted: str, aliases: list[str]) -> pd.DataFrame:
    """
    Ensure df has column `wanted` by renaming any alias found.
    """
    cols = {c.upper(): c for c in df.columns}
    if wanted.upper() in cols:
        return df
    for a in aliases:
        if a.upper() in cols:
            df.rename(columns={cols[a.upper()]: wanted}, inplace=True)
            return df
    return df

# ==========================================================
# Step 1 — Check for Required Workforce Files
# ==========================================================
#
# This step ensures that two workforce files required later in
# the pipeline are available in the processed folder.
#
# If they are missing from processed but present in raw, they
# are copied across automatically.
#
# This safeguard improves robustness when the project is
# rerun in a fresh environment.
# ==========================================================

# --- STEP 1: HUNT DOWN MISSING FILES ---
required_files = ['workforce_turnover.csv', 'workforce_ratios.csv']
for filename in required_files:
    proc_path = PROCESSED_FOLDER / filename
    raw_path = RAW_FOLDER / filename

    if proc_path.exists():
        print(f" Found {filename} in Processed.")
    elif raw_path.exists():
        print(f" Found {filename} in Raw... Copying to Processed.")
        shutil.copy(raw_path, proc_path)
    else:
        print(f" CRITICAL WARNING: Could not find {filename} in Raw or Processed!")
        print(f"   Please manually move {filename} to: {PROCESSED_FOLDER}")

# ==========================================================
# Step 2 — Define Dataset Categories
# ==========================================================
#
# Files are grouped into conceptual source categories so they
# can later be merged and accessed consistently.
#
# Categories used in this project:
# - census
# - absence
# - school_info
# - spine
# - swf
# - ks4
# - links
#
# The final cached object stores one combined DataFrame per
# category.
# ==========================================================

# --- STEP 2: DEFINE CATEGORIES ---
data_categories = {
    'census': [], 'absence': [], 'school_info': [], 'spine': [],
    'swf': [], 'ks4': [], 'links': []
}

# ==========================================================
# Step 3 — Ingest All Processed CSV Files
# ==========================================================
#
# This step scans the processed folder, reads each CSV, assigns
# or standardises academic year values, and appends the result
# to the appropriate category.
#
# Special handling is applied to workforce files:
# - workforce_turnover.csv
# - workforce_ratios.csv
#
# This is necessary because these files require additional
# schema harmonisation and metric derivation before they can
# be merged with the rest of the source data.
# ==========================================================

# --- STEP 3: INGEST ALL FILES ---
all_files = sorted(PROCESSED_FOLDER.glob("*.csv"))

print("\nProcessing Files...")
for filepath in all_files:
    filename = filepath.name.lower()

    try:
        # === HANDLER: TURNOVER ===
        if 'workforce_turnover' in filename:
            print(f"     Ingesting TURNOVER: {filename}")
            df = safe_read_csv(filepath)

            # Normalise column names for this file
            df = upper_cols(df)
            df = ensure_col(df, 'URN', ['SCHOOL_URN', 'URN'])
            df = ensure_col(df, 'ACADEMICYEAR', ['TIME_PERIOD', 'ACADEMIC_YEAR', 'ACADEMICYEAR'])

            # Standardise year
            df['AcademicYear'] = df['ACADEMICYEAR'].apply(standardise_academic_year)

            # Make numeric
            for c in ['LEFT_THE_STATE-FUNDED_SYSTEM', 'LEFT_TO_ANOTHER_STATE-FUNDED_SCHOOL', 'TEACHER_FTE_IN_CENSUS_YEAR']:
                if c in df.columns:
                    df[c] = pd.to_numeric(df[c], errors='coerce')

            # Calculate turnover
            leavers_a = df.get('LEFT_THE_STATE-FUNDED_SYSTEM', 0)
            leavers_b = df.get('LEFT_TO_ANOTHER_STATE-FUNDED_SCHOOL', 0)
            denom = df.get('TEACHER_FTE_IN_CENSUS_YEAR', pd.NA)

            df['Leavers'] = leavers_a.fillna(0) + leavers_b.fillna(0)
            df['TurnoverRate'] = (df['Leavers'] / denom) * 100
            df['TurnoverRate'] = df['TurnoverRate'].replace([pd.NA, float('inf')], 0).fillna(0)

            out = df[['URN', 'AcademicYear', 'TurnoverRate']].copy()
            out['URN'] = pd.to_numeric(out['URN'], errors='coerce')

            data_categories['swf'].append(out)
            continue

        # === HANDLER: RATIOS ===
        if 'workforce_ratios' in filename:
            print(f"     Ingesting RATIOS: {filename}")
            df = safe_read_csv(filepath)
            df = upper_cols(df)
            df = ensure_col(df, 'URN', ['SCHOOL_URN', 'URN'])
            df = ensure_col(df, 'ACADEMICYEAR', ['TIME_PERIOD', 'ACADEMIC_YEAR', 'ACADEMICYEAR'])

            df['AcademicYear'] = df['ACADEMICYEAR'].apply(standardise_academic_year)

            # rename potential columns
            if 'TEACHERS_FTE' in df.columns:
                df.rename(columns={'TEACHERS_FTE': 'TotalTeachers'}, inplace=True)
            if 'PUPIL_TO_QUAL_TEACHER_RATIO' in df.columns:
                df.rename(columns={'PUPIL_TO_QUAL_TEACHER_RATIO': 'PTR'}, inplace=True)

            for c in ['TotalTeachers', 'PTR']:
                if c in df.columns:
                    df[c] = pd.to_numeric(df[c], errors='coerce')

            out = df[['URN', 'AcademicYear'] + [c for c in ['TotalTeachers', 'PTR'] if c in df.columns]].copy()
            out['URN'] = pd.to_numeric(out['URN'], errors='coerce')

            data_categories['swf'].append(out)
            continue

        # === STANDARD FILES ===
        df_temp = safe_read_csv(filepath)
        df_temp = upper_cols(df_temp)

        # Add AcademicYear from filename if present
        extracted_year = extract_year_from_filename(filename)
        if extracted_year:
            df_temp['AcademicYear'] = extracted_year
        else:
            # If file already contains an academic year column, standardise it
            if 'ACADEMICYEAR' in df_temp.columns:
                df_temp['AcademicYear'] = df_temp['ACADEMICYEAR'].apply(standardise_academic_year)

        # Categorise by filename
        if 'links' in filename:
            data_categories['links'].append(df_temp)
        elif 'census' in filename:
            data_categories['census'].append(df_temp)
        elif 'abs' in filename:
            data_categories['absence'].append(df_temp)
        elif 'school_information' in filename:
            data_categories['school_info'].append(df_temp)
        elif 'spine' in filename:
            data_categories['spine'].append(df_temp)
        elif re.search(r'\bswf\b', filename):
            data_categories['swf'].append(df_temp)
        elif 'ks4' in filename:
            data_categories['ks4'].append(df_temp)

    except Exception as e:
        print(f"    Error loading {filename}: {e}")

# ==========================================================
# Step 4 — Merge Category-Level Data and Save Cache
# ==========================================================
#
# Each category is concatenated into a single DataFrame.
#
# Special handling is applied to SWF data because multiple
# workforce-related files may contain overlapping fields for
# the same school-year combination. These are grouped by:
# - URN
# - AcademicYear
#
# and combined using a first non-null aggregation rule.
#
# The final output is saved as a pickle cache for use in later
# notebooks.
# ==========================================================

# --- STEP 4: MERGE & SAVE ---
print("\nMerging Categories...")
master_dfs = {}

for cat, dfs in data_categories.items():
    if not dfs:
        continue

    if cat == 'swf':
        combined = pd.concat(dfs, ignore_index=True)

        # Clean keys
        combined['URN'] = pd.to_numeric(combined.get('URN'), errors='coerce')
        combined['AcademicYear'] = combined.get('AcademicYear').apply(standardise_academic_year)

        # Prefer “first non-null” per group rather than max()
        def first_nonnull(series):
            s = series.dropna()
            return s.iloc[0] if len(s) else pd.NA

        agg_cols = {c: first_nonnull for c in combined.columns if c not in ['URN', 'AcademicYear']}
        combined = combined.groupby(['URN', 'AcademicYear'], as_index=False).agg(agg_cols)

        master_dfs[cat] = combined
    else:
        master_dfs[cat] = pd.concat(dfs, ignore_index=True)

    print(f"    Merged {cat.upper()}: {len(master_dfs[cat]):,} rows")

# Save cache (DICT)
CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(CACHE_PATH, 'wb') as f:
    pickle.dump(master_dfs, f)

print(f"\n DONE. Cache saved to: {CACHE_PATH}")

# --- OPTIONAL: QUICK VERIFY ---
print("\n Cache Summary:")
for k, v in master_dfs.items():
    if isinstance(v, pd.DataFrame):
        print(f"   {k:<12} → {v.shape[0]:>8,} rows | {v.shape[1]} cols")

 Starting SMART Data Ingestion...

 Found workforce_turnover.csv in Processed.
 Found workforce_ratios.csv in Processed.

Processing Files...
     Ingesting RATIOS: workforce_ratios.csv
   🛠️  Ingesting TURNOVER: workforce_turnover.csv

Merging Categories...
    Merged CENSUS: 190,694 rows
    Merged ABSENCE: 422,573 rows
    Merged SCHOOL_INFO: 130,319 rows
    Merged SPINE: 52,406 rows


In [ ]:
# ==========================================================
# Optional Verification Cell — Inspect Cached Links Dataset
# ==========================================================
#
# Purpose:
# This cell performs a simple manual verification of the
# newly created cache by loading the pickle file and
# inspecting the 'links' dataset.
#
# Why this check is useful:
# The ingestion notebook combines multiple source files into
# a cached dictionary. This quick inspection confirms that
# the cache was written successfully and that the expected
# category structure is available for downstream use.
#
# Note:
# This is a diagnostic / QA cell and is not required for the
# core pipeline to run.
# ==========================================================


import pandas as pd

cache = pd.read_pickle(r"C:\Users\kiero\Documents\msc-dissertation\data\processed\master_data_cache.pkl")

print(cache["links"].columns.tolist())
print(cache["links"].head(3))

# ==========================================================
# Outputs Produced
# ==========================================================
#
# After successful execution, this notebook produces:
#
# - master_data_cache.pkl
#
# This cache contains the standardised category-level source
# datasets used by subsequent notebooks, including entity
# resolution, feature engineering, and model training.
#
# Next notebook in pipeline:
# - 02_academy mapper.ipynb
# ==========================================================

['URN', 'LINKURN', 'LINKNAME', 'LINKTYPE', 'LINKESTABLISHEDDATE']
      URN  LINKURN                       LINKNAME              LINKTYPE  \
0  100006   134643           CCfL Key Stage 3 PRU  Predecessor - merged   
1  100012   100021  Rhyl Community Primary School    Successor - merged   
2  100016   132245       Kingsgate Primary School             Successor   

  LINKESTABLISHEDDATE  
0          01-01-2022  
1          31-08-2021  
2                 NaN  
